mma in transformers

In [ ]:
import os
import re
import time
from collections import Counter
import pandas as pd
import cupy as cp
from sentence_transformers import SentenceTransformer
import numpy as np
import torch 

os.environ["CUPY_NVCC_GENERATE_CODE"] = "arch=compute_89,code=sm_89"
os.environ["PATH"] = r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\bin;" + os.environ["PATH"]

assert cp.cuda.runtime.getDeviceCount() > 0, "No CUDA GPU detected!"
with cp.cuda.Device(0) as dev:
    props = cp.cuda.runtime.getDeviceProperties(dev.id)
    print(f"[GPU] Using: {props['name'].decode()} (SMs={props['multiProcessorCount']})")

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^\w\s$]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def matmul_mma_fn(A, B):
    """Performs FP16 matrix multiplication using cuBLAS Tensor Cores."""
    A_half = A.astype(cp.float16)
    B_half = B.astype(cp.float16)
    C_half = cp.matmul(A_half, B_half)  
    return C_half.astype(cp.float32)



#gpu kernels

matmul_tiled_code = '''
extern "C" __global__
void matmul_tiled(const float* A, const float* B, float* C, int M, int K, int N) {
    __shared__ float sA[16][16];
    __shared__ float sB[16][16];

    int row = blockIdx.y * 16 + threadIdx.y;
    int col = blockIdx.x * 16 + threadIdx.x;

    float value = 0.0f;

    for (int t = 0; t < (K + 16 - 1)/16; t++) {
        int tiled_row = row;
        int tiled_col = t*16 + threadIdx.x;
        if(tiled_row < M && tiled_col < K) sA[threadIdx.y][threadIdx.x] = A[tiled_row*K + tiled_col];
        else sA[threadIdx.y][threadIdx.x] = 0.0f;

        tiled_row = t*16 + threadIdx.y;
        tiled_col = col;
        if(tiled_row < K && tiled_col < N) sB[threadIdx.y][threadIdx.x] = B[tiled_row*N + tiled_col];
        else sB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for(int k=0;k<16;k++) value += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }

    if(row < M && col < N) C[row*N + col] = value;
}
'''
matmul_tiled_kernel = cp.RawKernel(matmul_tiled_code, 'matmul_tiled')

def matmul_tiled(A, B):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, "Matrix inner dimensions must match."
    C = cp.zeros((M, N), dtype=cp.float32)
    block = (16, 16)
    grid = ((N + 15)//16, (M + 15)//16)
    matmul_tiled_kernel(grid, block, (A, B, C, np.int32(M), np.int32(K), np.int32(N)))
    return C

softmax_kernel_code = r'''
extern "C" __global__
void row_softmax(float* X, int M, int N){
    int row = blockDim.x*blockIdx.x + threadIdx.x;
    if(row<M){
        float max_val = -1e20f;
        for(int j=0;j<N;j++){
            float v = X[row*N+j];
            if(v>max_val) max_val=v;
        }
        float sum_exp=0.0f;
        for(int j=0;j<N;j++){
            float e = __expf(X[row*N+j]-max_val);
            X[row*N+j] = e;
            sum_exp += e;
        }
        float inv = 1.0f/(sum_exp+1e-9f);
        for(int j=0;j<N;j++) X[row*N+j]*=inv;
    }
}
'''
softmax_kernel = cp.RawKernel(softmax_kernel_code, 'row_softmax')

def softmax_manual(X):
    M, N = X.shape
    block = (128,1,1)
    grid = ((M+127)//128,1,1)
    softmax_kernel(grid, block, (X, np.int32(M), np.int32(N)))
    return X

relu_kernel_code = r'''
extern "C" __global__
void relu(float* X, int MN){
    int idx = blockDim.x*blockIdx.x + threadIdx.x;
    if(idx<MN){
        float v = X[idx];
        X[idx] = v>0.0f ? v : 0.0f;
    }
}
'''
relu_kernel = cp.RawKernel(relu_kernel_code, 'relu')

def relu_manual(X):
    MN = X.size
    block = (256,1,1)
    grid = ((MN+255)//256,1,1)
    relu_kernel(grid, block, (X, np.int32(MN)))
    return X

# Transformer Classifier
class TransformerClassifierGPU:
    def __init__(self, input_dim, n_classes, d_model=64, lr=0.06, matmul_mode='mma'):
        """
        matmul_mode: 'mma' -> cuBLAS Tensor Core
                     'tiled' -> custom tiled CUDA kernel
        """
        self.d_model = d_model
        self.n_classes = n_classes
        self.lr = lr
        self.matmul_mode = matmul_mode
        self.matmul = matmul_mma_fn if matmul_mode=='mma' else matmul_tiled

        limit = np.sqrt(6 / (input_dim + d_model))
        self.W_embed = cp.random.uniform(-limit, limit, (input_dim, d_model)).astype(cp.float16)
        self.W_Q = cp.random.uniform(-limit, limit, (d_model, d_model)).astype(cp.float16)
        self.W_K = cp.random.uniform(-limit, limit, (d_model, d_model)).astype(cp.float16)
        self.W_V = cp.random.uniform(-limit, limit, (d_model, d_model)).astype(cp.float16)
        self.W_ff = cp.random.uniform(-limit, limit, (d_model, d_model)).astype(cp.float16)
        self.b_ff = cp.zeros(d_model, dtype=cp.float32)
        self.W_out = cp.random.uniform(-limit, limit, (d_model, n_classes)).astype(cp.float16)
        self.b_out = cp.zeros(n_classes, dtype=cp.float32)

    def forward(self, X):
        X_h = X.astype(cp.float16)
     
        X_emb = self.matmul(X_h, self.W_embed).astype(cp.float32)
        Q = self.matmul(X_emb, self.W_Q.astype(cp.float32))
        K = self.matmul(X_emb, self.W_K.astype(cp.float32))
        V = self.matmul(X_emb, self.W_V.astype(cp.float32))
        scores = self.matmul(Q, K.T) / cp.sqrt(cp.float32(self.d_model))
        attn = softmax_manual(scores.copy())
        attn_out = self.matmul(attn, V)
        X_res = X_emb + 0.5 * attn_out
        mean = cp.mean(X_res, axis=1, keepdims=True)
        std = cp.std(X_res, axis=1, keepdims=True) + 1e-6
        X_norm = (X_res - mean) / std
        FF_pre = self.matmul(X_norm.astype(cp.float32), self.W_ff.astype(cp.float32)) + self.b_ff
        FF = relu_manual(FF_pre.copy())
        X_ff = X_norm + FF
        logits = self.matmul(X_ff, self.W_out.astype(cp.float32)) + self.b_out
        return logits, X_ff

    def backward(self, X_ff, logits, y_true):
        M = X_ff.shape[0]
        y_onehot = cp.zeros((M, self.n_classes), dtype=cp.float32)
        y_onehot[cp.arange(M), y_true] = 1.0
        grad_out = (logits - y_onehot) / M
        self.W_out -= self.lr * self.matmul(X_ff.T, grad_out).astype(cp.float16)
        self.b_out -= self.lr * cp.sum(grad_out, axis=0)

def oversample_gpu(X, y):
    counts = Counter(cp.asnumpy(y))
    max_count = max(counts.values())
    X_list, y_list = [X],[y]
    for label,count in counts.items():
        if count<max_count:
            idxs = cp.where(y==label)[0]
            reps = cp.random.choice(idxs, size=(max_count-count,), replace=True)
            X_list.append(X[reps])
            y_list.append(y[reps])
    return cp.concatenate(X_list, axis=0), cp.concatenate(y_list, axis=0)

def manual_split(X, y, val_ratio=0.15, seed=42):
    np.random.seed(seed)
    idx = np.arange(len(y))
    np.random.shuffle(idx)
    split = int(len(y)*(1-val_ratio))
    train_idx, val_idx = idx[:split], idx[split:]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

if __name__ == "__main__":
    cp.random.seed(42)
    np.random.seed(42)

    df = pd.read_csv(r"C:\Users\Pavani Akshaya\OneDrive\Desktop\FIN_DA~1.CSV")
    sentences = [clean_text(s) for s in df["Sentence"].astype(str).tolist()]

    def map_label(x):
        if pd.isna(x): return 1
        s = str(x).strip().lower()
        if s in ("positive","pos","p","2","+"): return 2
        if s in ("negative","neg","n","0","-"): return 0
        if s in ("neutral","neu","1"): return 1
        if s.isdigit():
            v = int(s)
            return 0 if v <= 0 else 1 if v == 1 else 2
        return 1

    labels = df["Sentiment"].apply(map_label).astype(int).to_numpy()

    #  Sentence embeddings
    model_sbert = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    embeddings = model_sbert.encode(
        sentences,
        convert_to_tensor=True,
        device='cuda',
        show_progress_bar=True
    )

    X_gpu = cp.asarray(embeddings.cpu().numpy(), dtype=cp.float16)
    y_gpu = cp.asarray(labels, dtype=cp.int32)

    X_gpu, y_gpu = oversample_gpu(X_gpu, y_gpu)
    X_np, y_np = cp.asnumpy(X_gpu), cp.asnumpy(y_gpu)
    X_tr, X_val, y_tr, y_val = manual_split(X_np, y_np, val_ratio=0.15)
    X_tr, X_val = cp.asarray(X_tr), cp.asarray(X_val)
    y_tr, y_val = cp.asarray(y_tr), cp.asarray(y_val)

    for mode in ['mma', 'tiled']:
        print(f"\n=== Training with {mode.upper()} matmul ===")
        model = TransformerClassifierGPU(X_gpu.shape[1], n_classes=3, d_model=64, lr=0.06, matmul_mode=mode)
        epochs = 100
        
        for ep in range(epochs):
            start = time.time()
            logits, X_ff = model.forward(X_tr)
            model.backward(X_ff, logits, y_tr)
            preds = cp.argmax(logits, axis=1)
            acc = float(cp.mean((preds==y_tr).astype(cp.float32)))*100
            elapsed = time.time()-start
            if (ep+1) % 10 == 0 or ep == 0:
                print(f"Epoch {ep+1}/{epochs} | Acc: {acc:.2f}% | Time: {elapsed:.2f}s")

        # --- Validation ---
        logits_val, _ = model.forward(X_val)
        preds_val = cp.argmax(logits_val, axis=1)
        acc_val = float(cp.mean((preds_val==y_val).astype(cp.float32)))*100
        print(f"[Validation] Accuracy ({mode.upper()}): {acc_val:.2f}%")


[GPU] Using: NVIDIA GeForce RTX 3060 Laptop GPU (SMs=30)


Batches: 100%|██████████| 183/183 [00:05<00:00, 33.07it/s]



=== Training with MMA matmul ===
Epoch 1/100 | Acc: 32.80% | Time: 0.03s
Epoch 10/100 | Acc: 49.07% | Time: 0.03s
Epoch 20/100 | Acc: 55.27% | Time: 0.03s
Epoch 30/100 | Acc: 58.75% | Time: 0.03s
Epoch 40/100 | Acc: 60.66% | Time: 0.03s
Epoch 50/100 | Acc: 61.28% | Time: 0.03s
Epoch 60/100 | Acc: 61.66% | Time: 0.03s
Epoch 70/100 | Acc: 62.27% | Time: 0.03s
Epoch 80/100 | Acc: 62.54% | Time: 0.20s
Epoch 90/100 | Acc: 62.50% | Time: 0.04s
Epoch 100/100 | Acc: 62.65% | Time: 0.03s
[Validation] Accuracy (MMA): 59.90%

=== Training with TILED matmul ===
Epoch 1/100 | Acc: 33.35% | Time: 0.04s
Epoch 10/100 | Acc: 33.35% | Time: 0.05s
Epoch 20/100 | Acc: 33.35% | Time: 0.05s
Epoch 30/100 | Acc: 33.35% | Time: 0.05s
Epoch 40/100 | Acc: 33.35% | Time: 0.05s
Epoch 50/100 | Acc: 33.35% | Time: 0.05s
Epoch 60/100 | Acc: 33.50% | Time: 0.05s
Epoch 70/100 | Acc: 33.50% | Time: 0.05s
Epoch 80/100 | Acc: 33.50% | Time: 0.05s
Epoch 90/100 | Acc: 33.50% | Time: 0.07s
Epoch 100/100 | Acc: 33.50% | Time